# Step 1 - Metadata + Quality Checks + Subject-Disjoint Split

This notebook includes alias-aware action checks for THETIS naming conventions.

Action folders in scope:
- backhand
- forehand_flat
- kick_service
- smash

Key improvement:
- Quality check now compares **canonical action names** instead of raw filename tokens, so `foreflat` ↔ `forehand_flat` and `serkick` ↔ `kick_service` are handled correctly.


## Clarifications
1. `repeat_id` is trial index (`s1/s2/s3`), not skill.
2. Skill label rule from THETIS README:
   - `p1` to `p31`: beginner
   - `p32` to `p55`: expert
3. Subject-disjoint split is mandatory to avoid leakage.
4. This notebook generates metadata from GitHub tree and file names only.


## Mapping in Step 1
Professor feedback alignment:
- Class imbalance is reported overall and by split.
- Subject-disjoint split and leakage report are exported.
- Stroke/action handling keeps both raw token and canonical action for transparent analysis.
- All outputs are file-traceable for reproducibility.


In [ ]:
!pip -q install pandas requests scikit-learn

import re
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.parse import quote
from sklearn.model_selection import train_test_split

print('Environment ready.')


In [ ]:
# Config
OWNER = 'THETIS-dataset'
REPO = 'dataset'
BRANCH = 'main'
ACTION_FOLDERS = ['backhand', 'forehand_flat', 'kick_service', 'smash']

SEED = 42
TEST_SUBJECT_RATIO = 0.20
VAL_SUBJECT_RATIO = 0.15

GITHUB_TOKEN = None  # Optional

OUT_DIR = Path('step1_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('OUT_DIR:', OUT_DIR)
print('ACTION_FOLDERS:', ACTION_FOLDERS)


In [ ]:
# Build metadata from GitHub tree
PATTERNS = [
    re.compile(r'^p(?P<subject_num>\d+)_(?P<stroke>[a-zA-Z0-9_]+)_s(?P<repeat_id>\d+)$'),
    re.compile(r'^p(?P<subject_num>\d+)(?P<stroke>[a-zA-Z0-9]+?)(?P<repeat_id>\d+)$'),
]

# THETIS token-to-folder canonical mapping
ACTION_CANONICAL_MAP = {
    'backhand': 'backhand',
    'foreflat': 'forehand_flat',
    'serkick': 'kick_service',
    'smash': 'smash',
}


def parse_name(stem):
    for pat in PATTERNS:
        m = pat.match(stem)
        if m:
            subject_num = int(m.group('subject_num'))
            skill_label = 'beginner' if subject_num <= 31 else ('expert' if subject_num <= 55 else 'unknown')
            stroke_token = m.group('stroke').lower()
            canonical_action = ACTION_CANONICAL_MAP.get(stroke_token, stroke_token)
            return {
                'subject_num': subject_num,
                'subject_id': f'p{subject_num}',
                'skill_label': skill_label,
                'skill_binary': 0 if skill_label == 'beginner' else (1 if skill_label == 'expert' else ''),
                'stroke_token': stroke_token,
                'canonical_action': canonical_action,
                'repeat_id': int(m.group('repeat_id')),
            }
    return None


def fetch_tree(owner, repo, branch, token=None):
    url = f'https://api.github.com/repos/{owner}/{repo}/git/trees/{quote(branch, safe="")}?recursive=1'
    headers = {'Accept': 'application/vnd.github+json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json().get('tree', [])


tree = fetch_tree(OWNER, REPO, BRANCH, GITHUB_TOKEN)
wanted_prefixes = [f'VIDEO_RGB/{a}/' for a in ACTION_FOLDERS]
rows = []

for node in tree:
    if node.get('type') != 'blob':
        continue
    path = node.get('path', '')
    if not path.lower().endswith('.avi'):
        continue
    if not any(path.startswith(pref) for pref in wanted_prefixes):
        continue

    action_folder = path.split('/')[1]
    filename = path.split('/')[-1]
    stem = filename[:-4]
    parsed = parse_name(stem)

    row = {
        'repo_path': path,
        'raw_url': f'https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/{path}',
        'video_name': filename,
        'subject_id': '',
        'subject_num': '',
        'skill_label': '',
        'skill_binary': '',
        'stroke_token': '',
        'canonical_action': '',
        'repeat_id': '',
        'action_folder': action_folder,
        'modality_dir': 'VIDEO_RGB',
        'valid_flag': 1,
        'parse_note': '',
        'blob_sha': node.get('sha', ''),
        'blob_size_bytes': node.get('size', ''),
    }

    if parsed is None:
        row['valid_flag'] = 0
        row['parse_note'] = 'filename_parse_failed'
    else:
        row.update(parsed)
        if row['skill_label'] == 'unknown':
            row['valid_flag'] = 0
            row['parse_note'] = 'subject_out_of_range'

    rows.append(row)

meta_df = pd.DataFrame(rows).sort_values(['action_folder','subject_num','repeat_id']).reset_index(drop=True)
meta_df.to_csv(OUT_DIR / 'metadata_rgb_multi4_remote.csv', index=False)
print('Saved metadata:', OUT_DIR / 'metadata_rgb_multi4_remote.csv')
print('Rows:', len(meta_df))
meta_df.head()


In [ ]:
# Data quality checks (alias-aware)
qc = {}
flags = []

qc['row_count'] = int(len(meta_df))
qc['valid_rows'] = int((meta_df['valid_flag'] == 1).sum())
qc['invalid_rows'] = int((meta_df['valid_flag'] == 0).sum())

# path pattern
path_ok = meta_df['repo_path'].str.contains(r'^VIDEO_RGB/[a-z_]+/p\d+_[a-zA-Z0-9_]+_s\d+\.avi$', regex=True)
qc['path_pattern_fail_rows'] = int((~path_ok).sum())
for idx in meta_df.index[~path_ok]:
    flags.append({'row_index': int(idx), 'issue': 'path_pattern_fail', 'repo_path': meta_df.at[idx, 'repo_path']})

# label check
expected_label = meta_df['subject_num'].apply(lambda x: 'beginner' if 1 <= int(x) <= 31 else ('expert' if 32 <= int(x) <= 55 else 'unknown'))
label_mismatch = meta_df['skill_label'] != expected_label
qc['label_mismatch_rows'] = int(label_mismatch.sum())
for idx in meta_df.index[label_mismatch]:
    flags.append({'row_index': int(idx), 'issue': 'label_mismatch', 'repo_path': meta_df.at[idx, 'repo_path']})

# canonical action vs folder check
canon_mismatch = meta_df['canonical_action'] != meta_df['action_folder']
qc['canonical_action_folder_mismatch_rows'] = int(canon_mismatch.sum())
for idx in meta_df.index[canon_mismatch]:
    flags.append({'row_index': int(idx), 'issue': 'canonical_action_folder_mismatch', 'repo_path': meta_df.at[idx, 'repo_path'], 'canonical_action': meta_df.at[idx, 'canonical_action'], 'action_folder': meta_df.at[idx, 'action_folder']})

# keep old raw token mismatch for transparency (expected for foreflat/serkick)
raw_token_mismatch = meta_df['stroke_token'] != meta_df['action_folder']
qc['raw_token_folder_mismatch_rows'] = int(raw_token_mismatch.sum())

# subject coverage
subjects = sorted(meta_df['subject_num'].dropna().astype(int).unique().tolist())
expected_subjects = list(range(1,56))
qc['subject_unique_count'] = int(meta_df['subject_id'].nunique())
qc['missing_subjects'] = sorted(set(expected_subjects) - set(subjects))
qc['extra_subjects'] = sorted(set(subjects) - set(expected_subjects))

# repeats per subject/action
rep_sets = meta_df.groupby(['subject_id','action_folder'])['repeat_id'].apply(lambda s: tuple(sorted(set(int(v) for v in s.tolist()))))
bad_rep = rep_sets[rep_sets != (1,2,3)]
qc['subject_action_bad_repeat_patterns'] = int(len(bad_rep))

# duplicates
qc['duplicate_repo_path_rows'] = int(meta_df['repo_path'].duplicated().sum())
qc['duplicate_raw_url_rows'] = int(meta_df['raw_url'].duplicated().sum())

qc['class_counts'] = {str(k): int(v) for k,v in meta_df['skill_label'].value_counts().to_dict().items()}
qc['action_counts'] = {str(k): int(v) for k,v in meta_df['action_folder'].value_counts().to_dict().items()}

pd.DataFrame(flags).to_csv(OUT_DIR / 'quality_flags.csv', index=False)
(OUT_DIR / 'quality_check_summary.json').write_text(json.dumps(qc, indent=2))

print('Saved QC files')
print(json.dumps({
    'row_count': qc['row_count'],
    'valid_rows': qc['valid_rows'],
    'label_mismatch_rows': qc['label_mismatch_rows'],
    'canonical_action_folder_mismatch_rows': qc['canonical_action_folder_mismatch_rows'],
    'raw_token_folder_mismatch_rows': qc['raw_token_folder_mismatch_rows'],
    'subject_unique_count': qc['subject_unique_count'],
}, indent=2))


In [ ]:
# Subject-disjoint split
clean_df = meta_df[meta_df['valid_flag'] == 1].copy().reset_index(drop=True)

subj_df = clean_df[['subject_id','skill_binary']].drop_duplicates().sort_values('subject_id').reset_index(drop=True)
subject_ids = subj_df['subject_id'].values
subject_labels = subj_df['skill_binary'].values

subj_train_val, subj_test = train_test_split(subject_ids, test_size=TEST_SUBJECT_RATIO, random_state=SEED, stratify=subject_labels)
train_val_labels = subj_df.set_index('subject_id').loc[subj_train_val,'skill_binary'].values
val_ratio_relative = VAL_SUBJECT_RATIO / (1 - TEST_SUBJECT_RATIO)
subj_train, subj_val = train_test_split(subj_train_val, test_size=val_ratio_relative, random_state=SEED, stratify=train_val_labels)

subj_train, subj_val, subj_test = set(subj_train), set(subj_val), set(subj_test)

clean_df['split'] = clean_df['subject_id'].map(lambda s: 'train' if s in subj_train else ('val' if s in subj_val else ('test' if s in subj_test else 'unknown')))

ltv = len(subj_train & subj_val)
ltt = len(subj_train & subj_test)
lvt = len(subj_val & subj_test)
assert ltv == 0 and ltt == 0 and lvt == 0

seq_stats = clean_df.groupby(['split','skill_label']).size().reset_index(name='sequence_count')
subj_stats = clean_df.groupby('split')['subject_id'].nunique().reset_index(name='subject_count')
action_stats = clean_df.groupby(['split','action_folder']).size().reset_index(name='sequence_count')

clean_df.to_csv(OUT_DIR / 'metadata_rgb_multi4_with_split.csv', index=False)
seq_stats.to_csv(OUT_DIR / 'split_sequence_stats.csv', index=False)
subj_stats.to_csv(OUT_DIR / 'split_subject_stats.csv', index=False)
action_stats.to_csv(OUT_DIR / 'split_action_stats.csv', index=False)

leak_report = f"Leakage report\ntrain_intersect_val: {ltv}\ntrain_intersect_test: {ltt}\nval_intersect_test: {lvt}\n"
(OUT_DIR / 'leakage_report.txt').write_text(leak_report)

manifest = {
    'seed': SEED,
    'action_folders': ACTION_FOLDERS,
    'test_subject_ratio': TEST_SUBJECT_RATIO,
    'val_subject_ratio': VAL_SUBJECT_RATIO,
    'subjects': {'train': sorted(subj_train), 'val': sorted(subj_val), 'test': sorted(subj_test)},
    'sequence_counts_by_split_and_skill': seq_stats.to_dict(orient='records'),
    'subject_counts_by_split': subj_stats.to_dict(orient='records'),
    'sequence_counts_by_split_and_action': action_stats.to_dict(orient='records')
}
(OUT_DIR / 'split_manifest.json').write_text(json.dumps(manifest, indent=2))

print('Saved split artifacts to', OUT_DIR)
print(seq_stats)
print(subj_stats)
print(action_stats)
print(leak_report)


## Handoff to Step 2
Use these files:
- `metadata_rgb_multi4_with_split.csv`
- `split_manifest.json`
- `quality_check_summary.json`
- `split_sequence_stats.csv`
- `split_subject_stats.csv`
- `split_action_stats.csv`
- `leakage_report.txt`


In [ ]:
note = """# Step 1 Handoff Note

## Key clarifications
- Raw filename action token may differ from folder naming (`foreflat` vs `forehand_flat`, `serkick` vs `kick_service`).
- Canonical action mapping is applied in this version.
- Use `canonical_action` or `action_folder` for modeling/report tables, not raw token only.

## Files produced
- metadata_rgb_multi4_remote.csv
- metadata_rgb_multi4_with_split.csv
- quality_check_summary.json
- quality_flags.csv
- split_sequence_stats.csv
- split_subject_stats.csv
- split_action_stats.csv
- split_manifest.json
- leakage_report.txt
"""
(OUT_DIR / 'step1_handoff_note.md').write_text(note)
print('Saved handoff note')


In [ ]:
import shutil
from pathlib import Path
from google.colab import files
zip_base = Path('step1_outputs')
zip_file = zip_base.with_suffix('.zip')
shutil.make_archive(str(zip_base), 'zip', OUT_DIR)
print('Created:', str(zip_file.resolve()))
files.download(str(zip_file))
